# Cardiac-LLM Indic Benchmark v3 — Kaggle Runner

- **Model size cap: <= 8B parameters** (hard-enforced; oversized models raise an error unless `force_size=True`).
- **Dataset types covered:** `mcq`, `qa`, `text_gen`, `dialogue`.
- **Metrics:** exact_match, token-level **precision/recall/F1 (macro & micro)**, MCQ classification **F1/precision/recall (micro/macro/weighted)** + **confusion matrix**, ROUGE-1/2/L, BLEU, METEOR, **BERTScore** (semantic similarity, multilingual), plus generation **latency & tokens/sec**.

**Changed in this revision:** the whole scoring engine now lives in a **single cell** (Cell 4) as plain functions - no more writing `cardiac_benchmark.py` to disk and invoking it with `!python ...`. You call `run_benchmark(model_name=...)` directly from later cells instead. This also avoids the earlier `argparse` error, since there's no `sys.argv` parsing at all anymore.

**Before running:** turn on GPU (Settings -> Accelerator -> GPU T4 x2) and Internet (Settings -> Internet -> On).

## 1. Install dependencies

In [1]:
!pip install -q transformers torch rouge-score nltk python-dotenv tqdm sacremoses scikit-learn bitsandbytes accelerate bert-score
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 107.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.7 MB/s eta 0:00:00


## 2. Check GPU

In [14]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: True
Device: Tesla T4


## 3. HF token (only needed for gated/private models)

Add via **Add-ons -> Secrets** in Kaggle (key: `HF_TOKEN`).

In [2]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded (ok)")
except Exception as e:
    print("No HF_TOKEN secret - continuing without it (fine for public models):", e)


HF_TOKEN loaded (ok)


## 4. Benchmark engine (single cell - define once, call many times)

Defines `run_benchmark(model_name, device="cuda", limit=0, batch_size=8, max_new_tokens=128, types=None, force_size=False)`.
Run this cell once per session; call `run_benchmark(...)` as many times as you like afterward (once per model).

In [11]:
"""
Cardiac-LLM Indic Benchmark — scoring engine (in-kernel version, no argparse/subprocess)
Constraints: models <= 8B params only. Dataset types: mcq, qa, text_gen, dialogue.
Metrics: exact_match, token-P/R/F1 (macro=mean-per-example, micro=pooled TP/FP/FN),
         classification F1/precision/recall (micro/macro/weighted) + confusion matrix
         for MCQ, ROUGE-1/2/L, BLEU, METEOR, BERTScore (semantic similarity),
         plus latency/throughput.

Call run_benchmark(model_name=..., ...) directly - no `!python ...` needed.
"""
import json, re, string, time, csv, os
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
)
from bert_score import score as bertscore_score

for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    try:
        nltk.data.find(pkg)
    except LookupError:
        nltk.download(pkg, quiet=True)

MAX_PARAMS = 8_000_000_000  # hard cap: 8B params

BENCH_DIR = Path("output/benchmark")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TASK_TYPES = ["mcq", "qa", "dialogue", "text_gen"]

# BERTScore backbone: multilingual so it works across Indic languages, not just English
BERTSCORE_MODEL = "bert-base-multilingual-cased"

# ---------------------------------------------------------------- text utils
def normalize_text(s: str) -> str:
    s = s or ""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def exact_match(pred: str, gold: str) -> int:
    return int(normalize_text(pred) == normalize_text(gold))

def token_prf(pred: str, gold: str):
    """SQuAD-style token overlap -> precision, recall, f1, tp, fp, fn"""
    p_toks = normalize_text(pred).split()
    g_toks = normalize_text(gold).split()
    common = Counter(p_toks) & Counter(g_toks)
    tp = sum(common.values())
    fp = len(p_toks) - tp
    fn = len(g_toks) - tp
    if tp == 0:
        return 0.0, 0.0, 0.0, tp, fp, fn
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1, tp, fp, fn

def extract_mcq_letter(text: str):
    text = (text or "").strip()
    m = re.search(r"\b([A-E])\b", text.upper())
    if m:
        return m.group(1)
    m = re.search(r"answer\s*[:\-]?\s*([A-E])", text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return text.strip()[:1].upper() if text else ""

_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
_smoothing = SmoothingFunction().method1

def rouge_scores(pred, gold):
    s = _rouge.score(gold, pred)
    return s["rouge1"].fmeasure, s["rouge2"].fmeasure, s["rougeL"].fmeasure

def bleu_score(pred, gold):
    ref = normalize_text(gold).split()
    hyp = normalize_text(pred).split()
    if not hyp or not ref:
        return 0.0
    return sentence_bleu([ref], hyp, smoothing_function=_smoothing)

def meteor(pred, gold):
    ref = normalize_text(gold).split()
    hyp = normalize_text(pred).split()
    if not hyp or not ref:
        return 0.0
    try:
        return meteor_score([ref], hyp)
    except Exception:
        return 0.0

def bertscore_batch(preds, golds, device):
    """Batched BERTScore over a whole task type — much faster than per-example calls."""
    safe_pairs = [(p if p.strip() else " ", g if g.strip() else " ") for p, g in zip(preds, golds)]
    preds_c = [p for p, _ in safe_pairs]
    golds_c = [g for _, g in safe_pairs]
    P, R, F1 = bertscore_score(
        preds_c, golds_c,
        model_type=BERTSCORE_MODEL,
        lang=None,
        rescale_with_baseline=False,
        device=device if "cuda" in device else "cpu",
        verbose=False,
    )
    return P.tolist(), R.tolist(), F1.tolist()

# ---------------------------------------------------------------- model
def load_model(model_name, device, force_size=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if "cuda" in device else torch.float32,
            device_map={"": 0} if "cuda" in device else None,
            trust_remote_code=True,
        )
    except OSError:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if "cuda" in device else torch.float32,
            device_map={"": 0} if "cuda" in device else None,
            trust_remote_code=True,
            use_safetensors=False,
        )

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model params: {n_params/1e9:.2f}B")
    if n_params > MAX_PARAMS and not force_size:
        raise RuntimeError(
            f"REJECTED: {model_name} has {n_params/1e9:.2f}B params — "
            f"benchmark only allows models <= 8B. Pass force_size=True to override."
        )
    model.eval()
    return tokenizer, model

def build_prompt(instruction, inp):
    if inp and str(inp).strip():
        return f"{instruction}\n\n{inp}\n\nAnswer:"
    return f"{instruction}\n\nAnswer:"

@torch.no_grad()
def generate_batch(tokenizer, model, device, prompts, max_new_tokens=128):
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    texts = []
    for i in range(out.shape[0]):
        gen_ids = out[i][enc["input_ids"].shape[1]:]
        texts.append(tokenizer.decode(gen_ids, skip_special_tokens=True).strip())
    return texts

# ---------------------------------------------------------------- eval loop
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def run_type(dtype, tokenizer, model, device, batch_size, limit, max_new_tokens):
    path = BENCH_DIR / f"cardiac_benchmark_{dtype}.jsonl"
    if not path.exists():
        print(f"  (skip) {path} not found")
        return None, [], None

    rows = load_jsonl(path)
    if limit:
        rows = rows[:limit]

    records = []
    t0 = time.time()
    total_new_tokens = 0

    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        prompts = [build_prompt(r.get("instruction", ""), r.get("input", "")) for r in chunk]
        preds = generate_batch(tokenizer, model, device, prompts, max_new_tokens)
        for r, pred in zip(chunk, preds):
            total_new_tokens += len(tokenizer.encode(pred))
            records.append({
                "type": dtype,
                "instruction": r.get("instruction", ""),
                "input": r.get("input", ""),
                "gold": r.get("output", ""),
                "pred": pred,
                "source": r.get("source", ""),
                "language": r.get("language", ""),
            })

    elapsed = time.time() - t0
    tok_per_sec = total_new_tokens / elapsed if elapsed > 0 else 0.0

    metrics, confusion = score_type(dtype, records, device)
    metrics["latency_sec"] = round(elapsed, 2)
    metrics["tokens_per_sec"] = round(tok_per_sec, 2)
    metrics["n"] = len(records)
    return metrics, records, confusion

def score_type(dtype, records, device):
    if not records:
        return {}, None

    if dtype == "mcq":
        y_true, y_pred = [], []
        for r in records:
            g = extract_mcq_letter(r["gold"])
            p = extract_mcq_letter(r["pred"])
            y_true.append(g)
            y_pred.append(p)
        labels = sorted(set(y_true) | set(y_pred))
        acc = accuracy_score(y_true, y_pred)
        f1_micro = f1_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        f1_macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        f1_weighted = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
        prec_macro = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        rec_macro = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        prec_micro = precision_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        rec_micro = recall_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)

        cm = confusion_matrix(y_true, y_pred, labels=labels)
        confusion = {"labels": labels, "matrix": cm.tolist()}

        metrics = {
            "accuracy": round(acc, 4),
            "exact_match": round(acc, 4),
            "f1_micro": round(f1_micro, 4),
            "f1_macro": round(f1_macro, 4),
            "f1_weighted": round(f1_weighted, 4),
            "precision_macro": round(prec_macro, 4),
            "recall_macro": round(rec_macro, 4),
            "precision_micro": round(prec_micro, 4),
            "recall_micro": round(rec_micro, 4),
        }
        return metrics, confusion

    # free-text types: qa, dialogue, text_gen
    ems, f1s, precs, recs, r1s, r2s, rLs, bleus, meteors = [], [], [], [], [], [], [], [], []
    tp_sum = fp_sum = fn_sum = 0
    preds_all, golds_all = [], []
    for r in records:
        gold, pred = r["gold"], r["pred"]
        preds_all.append(pred); golds_all.append(gold)
        ems.append(exact_match(pred, gold))
        p, rec, f1, tp, fp, fn = token_prf(pred, gold)
        precs.append(p); recs.append(rec); f1s.append(f1)
        tp_sum += tp; fp_sum += fp; fn_sum += fn
        r1, r2, rL = rouge_scores(pred, gold)
        r1s.append(r1); r2s.append(r2); rLs.append(rL)
        bleus.append(bleu_score(pred, gold))
        meteors.append(meteor(pred, gold))

    micro_p = tp_sum / (tp_sum + fp_sum) if (tp_sum + fp_sum) else 0.0
    micro_r = tp_sum / (tp_sum + fn_sum) if (tp_sum + fn_sum) else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0

    n = len(records)

    try:
        bs_p, bs_r, bs_f1 = bertscore_batch(preds_all, golds_all, device)
        bertscore_p = sum(bs_p) / n
        bertscore_r = sum(bs_r) / n
        bertscore_f1 = sum(bs_f1) / n
    except Exception as e:
        print(f"  BERTScore failed ({e}) — reporting 0.0")
        bertscore_p = bertscore_r = bertscore_f1 = 0.0

    return {
        "exact_match": round(sum(ems) / n, 4),
        "precision_macro": round(sum(precs) / n, 4),
        "recall_macro": round(sum(recs) / n, 4),
        "f1_macro": round(sum(f1s) / n, 4),        # mean of per-example token F1
        "precision_micro": round(micro_p, 4),
        "recall_micro": round(micro_r, 4),
        "f1_micro": round(micro_f1, 4),            # pooled TP/FP/FN across all examples
        "rouge1": round(sum(r1s) / n, 4),
        "rouge2": round(sum(r2s) / n, 4),
        "rougeL": round(sum(rLs) / n, 4),
        "bleu": round(sum(bleus) / n, 4),
        "meteor": round(sum(meteors) / n, 4),
        "bertscore_precision": round(bertscore_p, 4),
        "bertscore_recall": round(bertscore_r, 4),
        "bertscore_f1": round(bertscore_f1, 4),
    }, None

# ---------------------------------------------------------------- main entry point
def run_benchmark(model_name, device="cuda", limit=0, batch_size=8,
                   max_new_tokens=128, types=None, force_size=False, save_results=True):
    """Call this directly from a notebook cell instead of `!python cardiac_benchmark.py ...`.

    save_results=False (use for sanity/smoke-test runs with a small `limit`):
    prints metrics to stdout but does NOT touch leaderboard.csv, benchmark_report.txt,
    or benchmark_results_<model>.csv - so quick test runs never pollute your real results.
    """
    types = types or TASK_TYPES

    print(f"\nModel   : {model_name}")
    print(f"Device  : {device}")
    print(f"Types   : {types}")
    print(f"Limit   : {limit or 'ALL'}")
    print(f"Saving  : {'yes' if save_results else 'no (sanity-check mode)'}\n")

    print(f"Loading model: {model_name}")
    tokenizer, model = load_model(model_name, device, force_size)

    all_metrics = {}
    all_records = []
    confusions = {}
    for dtype in types:
        print(f"\n--- Running {dtype} ---")
        metrics, records, confusion = run_type(dtype, tokenizer, model, device,
                                                batch_size, limit, max_new_tokens)
        if metrics:
            all_metrics[dtype] = metrics
            all_records.extend(records)
            print(f"  {dtype}: {metrics}")
        if confusion:
            confusions[dtype] = confusion

    if not save_results:
        print("\n(sanity-check mode: nothing written to results/ - rerun with save_results=True, "
              "or just use the full-run cell, once you're happy)")
        return all_metrics

    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name)

    # -- write per-model row-level CSV
    if all_records:
        with open(RESULTS_DIR / f"benchmark_results_{safe_name}.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(all_records[0].keys()))
            writer.writeheader()
            writer.writerows(all_records)

    # -- write confusion matrices (one CSV per type that has one, e.g. mcq)
    for dtype, conf in confusions.items():
        labels = conf["labels"]
        matrix = conf["matrix"]
        cm_path = RESULTS_DIR / f"confusion_matrix_{dtype}_{safe_name}.csv"
        with open(cm_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["true\\pred"] + labels)
            for label, row in zip(labels, matrix):
                writer.writerow([label] + row)
        print(f"Wrote confusion matrix: {cm_path}")

    # -- append to leaderboard.csv
    lb_path = RESULTS_DIR / "leaderboard.csv"
    rows_out = []
    for dtype, m in all_metrics.items():
        row = {"model": model_name, "type": dtype, **m}
        rows_out.append(row)

    if rows_out:
        all_keys = sorted({k for row in rows_out for k in row.keys()}, key=lambda k: (k != "model", k != "type", k))
        write_header = not lb_path.exists()
        with open(lb_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=all_keys)
            if write_header:
                writer.writeheader()
            for row in rows_out:
                writer.writerow(row)

    # -- human readable report (overwritten each run)
    with open(RESULTS_DIR / "benchmark_report.txt", "w", encoding="utf-8") as f:
        f.write(f"Cardiac-LLM Indic Benchmark Report\nModel: {model_name}\n")
        f.write("=" * 60 + "\n")
        for dtype, m in all_metrics.items():
            f.write(f"\n[{dtype}] (n={m.get('n')})\n")
            for k, v in m.items():
                if k != "n":
                    f.write(f"  {k:20s}: {v}\n")
        if confusions:
            f.write("\n" + "=" * 60 + "\n")
            f.write("Confusion matrices written as separate CSVs (see results/confusion_matrix_<type>_<model>.csv)\n")

    print("\nDone. Results written to results/")
    return all_metrics


## 5. Copy your benchmark data into the expected folder structure

Update `SRC` below to match your actual attached dataset's path. If unsure, run this cell once -
if it prints "No files found", it will also list every `.jsonl` Kaggle can actually see under `/kaggle/input`,
so you can copy the exact real path from there.

In [12]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/datasets/rounitsrivastava/cardi-b")
DST = Path("output/benchmark")
DST.mkdir(parents=True, exist_ok=True)

# maps your actual filenames -> the names run_benchmark() expects
FILENAME_MAP = {
    "mcq":      "mcq_hard_25.jsonl",
    "qa":       "qa_hard_25.jsonl",
    "dialogue": "dialogue_hard_25.jsonl",
    "text_gen": "text_gen_hard_25.jsonl",
}

copied = []
for dtype, fname in FILENAME_MAP.items():
    src_file = SRC / fname
    if src_file.exists():
        dst_file = DST / f"cardiac_benchmark_{dtype}.jsonl"
        shutil.copy(src_file, dst_file)
        copied.append(f"{fname} -> {dst_file.name}")
    else:
        print(f"  MISSING: {src_file}")

if copied:
    print("Copied:")
    for name in copied:
        print(" -", name)
else:
    print(f"No files found at {SRC} - check the dataset is attached and path is correct.")
    print("Attached inputs:")
    for p in Path("/kaggle/input").rglob("*.jsonl"):
        print(" ", p)

Copied:
 - mcq_hard_25.jsonl -> cardiac_benchmark_mcq.jsonl
 - qa_hard_25.jsonl -> cardiac_benchmark_qa.jsonl
 - dialogue_hard_25.jsonl -> cardiac_benchmark_dialogue.jsonl
 - text_gen_hard_25.jsonl -> cardiac_benchmark_text_gen.jsonl


In [13]:
import json
from pathlib import Path
from collections import Counter

# Automatically find the exact folder containing your files
matches = list(Path("/kaggle/input").rglob("mcq_hard_25.jsonl"))

if matches:
    SRC = matches[0].parent
    print(f"Found dataset at: {SRC}\n")
else:
    SRC = Path("/kaggle/input/cardi-b")
    print(f"Warning: Could not auto-locate files. Checking {SRC}\n")

FILES = {
    "mcq":      SRC / "mcq_hard_25.jsonl",
    "qa":       SRC / "qa_hard_25.jsonl",
    "dialogue": SRC / "dialogue_hard_25.jsonl",
    "text_gen": SRC / "text_gen_hard_25.jsonl",
}

print("=" * 70)
print("DATASET VALIDATION REPORT")
print("=" * 70)

overall_ok = True
for dtype, path in FILES.items():
    p = Path(path)
    print(f"\n--- {dtype.upper()} ---")
    if not p.exists():
        print(f"  FILE NOT FOUND: {path}")
        overall_ok = False
        continue

    total = empty_instruction = empty_input = empty_output = bad_json = 0
    sources, langs = Counter(), Counter()
    sample_row = None

    with open(p, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                bad_json += 1
                continue
            total += 1
            if sample_row is None:
                sample_row = row
            if not str(row.get("instruction") or "").strip():
                empty_instruction += 1
            if not str(row.get("input") or "").strip():
                empty_input += 1
            if not str(row.get("output") or "").strip():
                empty_output += 1
            sources[row.get("source", "unknown")] += 1
            langs[row.get("language", "unknown")] += 1

    print(f"  Total rows         : {total}")
    print(f"  Bad JSON lines     : {bad_json}")
    print(f"  Empty instruction  : {empty_instruction} ({empty_instruction/max(total,1)*100:.1f}%)")
    print(f"  Empty input        : {empty_input} ({empty_input/max(total,1)*100:.1f}%)")
    print(f"  Empty output (GOLD): {empty_output} ({empty_output/max(total,1)*100:.1f}%)")
    print(f"  Languages          : {len(langs)} -> {dict(langs.most_common())}")
    print(f"  Sources            : {len(sources)}")

    if empty_output > 0:
        overall_ok = False
        print(f"  WARNING: {empty_output} rows have NO gold answer - these will always score 0")

if overall_ok:
    print("\nDATASET LOOKS CLEAN - 0 scores are likely a model/prompting issue, not data.")
else:
    print("\nDATASET HAS ISSUES - fix these before blaming the model for 0 scores.")

Found dataset at: /kaggle/input/datasets/rounitsrivastava/cardi-b

DATASET VALIDATION REPORT

--- MCQ ---
  Total rows         : 25
  Bad JSON lines     : 0
  Empty instruction  : 0 (0.0%)
  Empty input        : 0 (0.0%)
  Empty output (GOLD): 0 (0.0%)
  Languages          : 1 -> {'en': 25}
  Sources            : 3

--- QA ---
  Total rows         : 25
  Bad JSON lines     : 0
  Empty instruction  : 0 (0.0%)
  Empty input        : 25 (100.0%)
  Empty output (GOLD): 0 (0.0%)
  Languages          : 1 -> {'en': 25}
  Sources            : 1

--- DIALOGUE ---
  Total rows         : 25
  Bad JSON lines     : 0
  Empty instruction  : 0 (0.0%)
  Empty input        : 0 (0.0%)
  Empty output (GOLD): 0 (0.0%)
  Languages          : 1 -> {'en': 25}
  Sources            : 1

--- TEXT_GEN ---
  Total rows         : 25
  Bad JSON lines     : 0
  Empty instruction  : 0 (0.0%)
  Empty input        : 0 (0.0%)
  Empty output (GOLD): 0 (0.0%)
  Languages          : 1 -> {'en': 25}
  Sources            : 1

## 6. Sanity-check run (small `limit`, fast, does NOT write to results/)

Model size is capped at **8B**. Use the full HuggingFace repo id, including the org prefix
(e.g. `google/gemma-3-4b-it`, not `gemma-3-4b-it`; `Qwen/Qwen3-4B`; `BioMistral/BioMistral-7B`).

`save_results=False` means this run only prints metrics to confirm everything works - it will
NOT touch `leaderboard.csv`, `benchmark_report.txt`, or `benchmark_results_<model>.csv`, so it can
never pollute your real results with `n=10` sanity-check numbers.

In [18]:
run_benchmark(model_name="microsoft/Phi-3-mini-4k-instruct", device="cuda", limit=10, batch_size=4, save_results=False)


Model   : microsoft/Phi-3-mini-4k-instruct
Device  : cuda
Types   : ['mcq', 'qa', 'dialogue', 'text_gen']
Limit   : 10
Saving  : no (sanity-check mode)

Loading model: microsoft/Phi-3-mini-4k-instruct


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

KeyError: 'type'

## 8. Full run

Runs all 4 dataset types over the whole benchmark set (no `limit`).

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instructt"  # change this per run (must be <= 8B params)

run_benchmark(model_name=MODEL_NAME, device="cuda", batch_size=4)

## 9. View this run's report

In [23]:
print(open("results/benchmark_report.txt").read())

Cardiac-LLM Indic Benchmark Report
Model: google/medgemma-1.5-4b-it

[mcq] (n=25)
  accuracy            : 0.0
  exact_match         : 0.0
  f1_micro            : 0.0
  f1_macro            : 0.0
  f1_weighted         : 0.0
  precision_macro     : 0.0
  recall_macro        : 0.0
  precision_micro     : 0.0
  recall_micro        : 0.0
  latency_sec         : 63.45
  tokens_per_sec      : 0.39

[qa] (n=25)
  exact_match         : 0.0
  precision_macro     : 0.0
  recall_macro        : 0.0
  f1_macro            : 0.0
  precision_micro     : 0.0
  recall_micro        : 0.0
  f1_micro            : 0.0
  rouge1              : 0.0
  rouge2              : 0.0
  rougeL              : 0.0
  bleu                : 0.0
  meteor              : 0.0
  bertscore_precision : 0.0
  bertscore_recall    : 0.0
  bertscore_f1        : 0.0
  latency_sec         : 62.84
  tokens_per_sec      : 0.4

[dialogue] (n=25)
  exact_match         : 0.0
  precision_macro     : 0.0
  recall_macro        : 0.0
  f1_macro   

## 10. Leaderboard (all models run so far, side by side)

`leaderboard.csv` appends a row per `(model, type)` every run, so re-running different models builds a comparison table.

In [24]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

leaderboard = pd.read_csv("results/leaderboard.csv")
leaderboard

,model,type,accuracy,bertscore_f1,bertscore_precision,bertscore_recall,bleu,exact_match,f1_macro,f1_micro,f1_weighted,latency_sec,meteor,n,precision_macro,precision_micro,recall_macro,recall_micro,rouge1,rouge2,rougeL,tokens_per_sec
0,deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B,mcq,0.08,NaN,NaN,NaN,NaN,0.08,0.0303,0.0800,0.04,33.81,NaN,25,0.0202,0.0800,0.0606,0.0800,NaN,NaN,NaN,88.13
1,deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B,qa,NaN,0.6459,0.6525,0.6399,0.0350,0.00,0.2243,0.2239,NaN,33.52,0.1440,25,0.2983,0.2972,0.1899,0.1796,0.2831,0.0771,0.1650,92.93
2,deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B,dialogue,NaN,0.5273,0.5414,0.5145,0.0025,0.00,0.0873,0.0900,NaN,36.01,0.0416,25,0.2745,0.2733,0.0540,0.0539,0.1076,0.0290,0.0761,78.87
3,deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B,text_gen,NaN,0.5940,0.5961,0.5963,0.0037,0.00,0.0954,0.0883,NaN,34.54,0.0642,25,0.1947,0.2005,0.1221,0.0566,0.1259,0.0166,0.0817,86.75
4,google/medgemma-1.5-4b-it,mcq,0.00,NaN,NaN,NaN,NaN,0.00,0.0000,0.0000,0.00,63.45,NaN,25,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,0.39
5,google/medgemma-1.5-4b-it,qa,NaN,0.0000,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,NaN,62.84,0.0000,25,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.40
6,google/medgemma-1.5-4b-it,dialogue,NaN,0.0000,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,NaN,68.96,0.0000,25,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.36
7,google/medgemma-1.5-4b-it,text_gen,NaN,0.0000,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,NaN,66.49,0.0000,25,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.38


## 11. Pivot view - one row per model, metrics side by side per task type

In [22]:
pivot = leaderboard.pivot_table(
    index="model",
    columns="type",
    values=["accuracy", "exact_match",
            "f1_micro", "f1_macro", "f1_weighted",
            "precision_micro", "precision_macro",
            "recall_micro", "recall_macro",
            "rouge1", "rouge2", "rougeL", "bleu", "meteor",
            "bertscore_precision", "bertscore_recall", "bertscore_f1",
            "tokens_per_sec"],
    aggfunc="last"
)
pivot

accuracy bertscore_f1                  bertscore_precision                  bertscore_recall                      bleu                  exact_match                      \
type                        mcq     dialogue      qa text_gen            dialogue      qa text_gen         dialogue      qa text_gen dialogue      qa text_gen    dialogue   mcq   qa text_gen   
model                                                                                                                                                                                            
Qwen/Qwen3.5-4B            0.12       0.6088  0.6769   0.6241              0.6228  0.6797   0.6212           0.5962  0.6748   0.6323   0.0065  0.0456   0.0142         0.0  0.12  0.0      0.0   
THUDM/glm-edge-4b-chat     0.12       0.6673  0.7289   0.6532              0.6970  0.7406   0.6502           0.6404  0.7181   0.6629   0.0070  0.0825   0.0145         0.0  0.12  0.0      0.0   

                       f1_macro                          f1_micro                        f1_weighted   meteor                  precision_macro                          precision_micro                \
type                   dialogue     mcq      qa text_gen dialogue   mcq      qa text_gen         mcq dialogue      qa text_gen        dialogue     mcq      qa text_gen        dialogue   mcq      qa   
model                                                                                                                                                                                                   
Qwen/Qwen3.5-4B          0.1328  0.0208  0.2480   0.1436   0.1347  0.12  0.2561   0.1056        0.03   0.0708  0.1789   0.1178          0.3663  0.0119  0.3096   0.2363          0.3868  0.12  0.3317   
THUDM/glm-edge-4b-chat   0.2174  0.0444  0.3426   0.1733   0.2175  0.12  0.3392   0.1446        0.08   0.1114  0.2431   0.1440          0.5988  0.0501  0.4101   0.2823          0.5978  0.12  0.4114   

                                recall_macro                          recall_micro                          rouge1                    rouge2                    rougeL                   \
type                   text_gen     dialogue     mcq      qa text_gen     dialogue   mcq      qa text_gen dialogue      qa text_gen dialogue      qa text_gen dialogue      qa text_gen   
model                                                                                                                                                                                     
Qwen/Qwen3.5-4B          0.2446       0.0817  0.0833  0.2202   0.1868       0.0816  0.12  0.2085   0.0673   0.1660  0.2938   0.1720   0.0660  0.1010   0.0389   0.1121  0.1675   0.1179   
THUDM/glm-edge-4b-chat   0.2939       0.1331  0.0899  0.3090   0.2461       0.1329  0.12  0.2886   0.0959   0.2283  0.4025   0.2028   0.1051  0.1589   0.0535   0.1477  0.2431   0.1235   

                       tokens_per_sec                         
type                         dialogue    mcq     qa text_gen  
model                                                         
Qwen/Qwen3.5-4B                 40.49  47.21  49.05    39.31  
THUDM/glm-edge-4b-chat          39.71  53.05  56.25    46.95

## 12. MCQ confusion matrix

Shows exactly which option letters the model confuses (rows = true gold letter, columns = predicted letter).

In [23]:
import re
from pathlib import Path
safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", MODEL_NAME)
cm_path = f"results/confusion_matrix_mcq_{safe_name}.csv"
if Path(cm_path).exists():
    cm_df = pd.read_csv(cm_path, index_col=0)
    display(cm_df)
else:
    print(f"No confusion matrix found at {cm_path} (did the mcq type run?)")

,A,B,C,D,E,I,L,P,T
true\pred,,,,,,,,,
A,2,0,0,1,0,0,0,0,0
B,3,0,0,0,0,0,0,0,0
C,5,0,1,0,0,0,0,0,1
D,1,1,0,0,0,0,0,0,0
E,0,0,0,0,0,0,0,0,1
I,4,0,0,0,0,0,0,0,0
L,1,0,0,0,0,0,0,0,0
P,1,0,0,0,0,0,0,0,1
T,0,0,2,0,0,0,0,0,0


## 13. Row-level predictions vs. gold for this model

In [21]:
row_results = pd.read_csv(f"results/benchmark_results_{safe_name}.csv")
row_results.head(20)

,type,instruction,input,gold,pred,source,language
0,mcq,A 71-year-old woman with a past medical histor...,"{""A"": ""Insert two large-bore intravenous cathe...",Initiate dopamine therapy and diuresis,A\n\nExplanation: The patient's condition has ...,ChuGyouk/MedQA,en
1,mcq,A 66-year-old female with hypertension and a r...,"{""A"": ""Intra-aortic balloon counterpulsation"",...",Immediate transfer to the operating room,C\n\nThe patient's presentation of sudden onse...,ChuGyouk/MedQA,en
2,mcq,A 58-year-old man presents to the emergency de...,"{""A"": ""Aortic dissection"", ""B"": ""Pulmonary emb...",Aortic dissection,Aortic dissection\n\nExplanation: The patient'...,ChuGyouk/MedQA,en
3,mcq,A 55-year-old man is brought to the emergency ...,"{""A"": ""Kawasaki disease"", ""B"": ""Rheumatic feve...",Cardiac tamponade,A\n\nExplanation: The patient's clinical prese...,maximegmd/MetaMedQA,en
4,mcq,A 55-year-old man is brought to the emergency ...,"{'A': 'Kawasaki disease', 'B': 'Rheumatic feve...",Cardiac tamponade,A\n\nExplanation: The patient's clinical prese...,maximegmd/MedQA-USMLE-4-options-clean,en
5,mcq,A 65-year-old man is brought to the emergency ...,"{""A"": ""Nifedipine"", ""B"": ""Enoxaparin"", ""C"": ""C...",Clopidogrel,C\n\nExplanation: Clopidogrel is an antiplatel...,maximegmd/MetaMedQA,en
6,mcq,A 65-year-old man is brought to the emergency ...,"{'A': 'Nifedipine', 'B': 'Enoxaparin', 'C': 'C...",Clopidogrel,C\n\nExplanation: Clopidogrel is an antiplatel...,maximegmd/MedQA-USMLE-4-options-clean,en
7,mcq,A 32-year-old man comes to the emergency depar...,"{""A"": ""Papillary muscle rupture"", ""B"": ""Pulmon...",Cardiac tamponade,C\n\nExplanation: The patient's presentation o...,ChuGyouk/MedQA,en
8,mcq,A 54-year-old man comes to the emergency depar...,"{""A"": ""Diffuse concave ST elevation and PR dep...",Low voltage and beat-to-beat variations in the...,"The patient's clinical presentation, including...",ChuGyouk/MedQA,en
9,mcq,Three days after undergoing cardiac catheteriz...,"{""A"": ""Ventricular septal rupture"", ""B"": ""Post...",Papillary muscle rupture,"B\n\nExplanation: The patient's symptoms, incl...",ChuGyouk/MedQA,en


## 14. Download everything

All files are under `/kaggle/working/results/`:
- `benchmark_report.txt`
- `leaderboard.csv` (cumulative across every model run this session)
- `benchmark_results_<model>.csv` (per-row detail, one file per model)
- `confusion_matrix_mcq_<model>.csv` (MCQ option-confusion breakdown, one file per model)

In [24]:
import shutil
shutil.make_archive("cardiac_benchmark_results", "zip", "results")
print("Created cardiac_benchmark_results.zip - download it from the Output panel.")

Created cardiac_benchmark_results.zip - download it from the Output panel.


## 15. GPU check (optional)

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)